# 01 · Demo: Token-Token Attention Heatmaps

This notebook demonstrates how to extract and visualize token-level attention
weights from a BERT-base model using Plotly interactive heatmaps.

**Install dependencies** (run once in Colab / fresh environment):
```bash
pip install -q transformers torch bertviz plotly
```

In [ ]:
# Uncomment to install in Colab
# !pip install -q transformers torch bertviz plotly

In [ ]:
import sys
import os

# Allow importing from src/ whether running locally or in Colab
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from extract_attention import (
    get_attention,
    get_mean_attention_per_layer,
    get_mean_attention_all_layers,
)

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import torch

## 1 · Extract attention weights

In [ ]:
MODEL_NAME = "bert-base-uncased"
SENTENCE = "The cat sat on the mat."

tokens, attentions = get_attention(SENTENCE, model_name=MODEL_NAME)

num_layers, num_heads, seq_len, _ = attentions.shape
print(f"Tokens ({seq_len}): {tokens}")
print(f"Attention tensor shape: {list(attentions.shape)}")

## 2 · Mean attention across all layers and heads

In [ ]:
mean_attn = get_mean_attention_all_layers(attentions).numpy()

fig = go.Figure(
    data=go.Heatmap(
        z=mean_attn,
        x=tokens,
        y=tokens,
        colorscale="Blues",
        zmin=0,
        zmax=mean_attn.max(),
        hoverongaps=False,
    )
)
fig.update_layout(
    title=f"{MODEL_NAME} — mean attention over all layers & heads",
    xaxis_title="Key token",
    yaxis_title="Query token",
    yaxis_autorange="reversed",
    width=600,
    height=550,
)
fig.show()

## 3 · Per-layer mean attention (grid)

In [ ]:
per_layer = get_mean_attention_per_layer(attentions)  # (num_layers, seq, seq)

# Show first 6 layers in a 2×3 grid
n_show = min(6, num_layers)
n_cols = 3
n_rows = (n_show + n_cols - 1) // n_cols

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Layer {i}" for i in range(n_show)],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

for i in range(n_show):
    row = i // n_cols + 1
    col = i % n_cols + 1
    layer_mat = per_layer[i].numpy()
    fig.add_trace(
        go.Heatmap(
            z=layer_mat,
            x=tokens,
            y=tokens,
            colorscale="Blues",
            showscale=(i == 0),
            zmin=0,
            zmax=layer_mat.max(),
        ),
        row=row,
        col=col,
    )

fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title=f"{MODEL_NAME} — mean attention per layer (first {n_show})",
    height=n_rows * 300,
    width=1000,
)
fig.show()

## 4 · Single layer, all heads

In [ ]:
LAYER = 5  # 0-indexed

n_cols = 4
n_rows = (num_heads + n_cols - 1) // n_cols

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Head {h}" for h in range(num_heads)],
    horizontal_spacing=0.06,
    vertical_spacing=0.10,
)

for h in range(num_heads):
    row = h // n_cols + 1
    col = h % n_cols + 1
    head_mat = attentions[LAYER, h].numpy()
    fig.add_trace(
        go.Heatmap(
            z=head_mat,
            x=tokens,
            y=tokens,
            colorscale="Blues",
            showscale=(h == 0),
            zmin=0,
            zmax=head_mat.max(),
        ),
        row=row,
        col=col,
    )

fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title=f"{MODEL_NAME} — Layer {LAYER}, all {num_heads} heads",
    height=n_rows * 280,
    width=1100,
)
fig.show()

## 5 · BertViz head view (HTML widget)

In [ ]:
try:
    from bertviz import head_view
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
    model.eval()

    inputs = tokenizer(SENTENCE, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    head_view(outputs.attentions, tokens)
except ImportError:
    print("bertviz not installed — run: pip install bertviz")